# argentina.matching — Pruebas interactivas

Matching difuso (fuzzy) sobre los catálogos del paquete. Útil cuando `lookup()` exacto falla con datos sucios.

## 1. Setup

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")

argentina v0.3.0


## 2. Provincias

`match_provincia` intenta primero `lookup()` exacto y solo si falla recurre a similitud.

In [2]:
arg.matching.match_provincia("buennos aires")

Provincia(nombre='Buenos Aires', codigo_indec='06', iso_id='AR-B', region='Pampeana', capital='La Plata', capital_lat=-34.9214, capital_lon=-57.9544, poblacion_2022=17569053, superficie_km2=307571)

In [3]:
arg.matching.match_provincia("cordova")

Provincia(nombre='Córdoba', codigo_indec='14', iso_id='AR-X', region='Pampeana', capital='Córdoba', capital_lat=-31.4201, capital_lon=-64.1888, poblacion_2022=3840905, superficie_km2=165321)

In [4]:
arg.matching.match_provincia("sgo del estero")

Provincia(nombre='Santiago del Estero', codigo_indec='86', iso_id='AR-G', region='NOA', capital='Santiago del Estero', capital_lat=-27.7951, capital_lon=-64.2615, poblacion_2022=1054028, superficie_km2=136351)

In [5]:
# Sin match: devuelve None
arg.matching.match_provincia("xyz")

### Top-N candidatos con score

Cuando quiero entender por qué cierto input matchea (o no).

In [6]:
for prov, score in arg.matching.candidatos_provincia("cordova", n=5):
    print(f"{score:.3f}  {prov.nombre}")

0.857  Córdoba
0.571  Formosa
0.429  Mendoza
0.375  Catamarca
0.375  Río Negro


## 3. Departamentos

Los nombres se repiten entre provincias. Filtrar por provincia evita ambigüedad.

In [7]:
arg.matching.match_departamento("gral san martin", provincia="Buenos Aires")

Departamento(codigo_departamento='06371', nombre='General San Martín', provincia_codigo='06', provincia_nombre='Buenos Aires')

In [8]:
# La provincia acepta cualquier alias que entienda arg.provincias.lookup
arg.matching.match_departamento("gral san martin", provincia="PBA")

Departamento(codigo_departamento='06371', nombre='General San Martín', provincia_codigo='06', provincia_nombre='Buenos Aires')

In [9]:
# Top-3 candidatos dentro de BA
for d, s in arg.matching.candidatos_departamento("san martin", provincia="Buenos Aires", n=3):
    print(f"{s:.3f}  {d.nombre}")

0.714  General San Martín
0.636  San Cayetano
0.600  La Matanza


## 4. Ciudades

In [10]:
arg.matching.match_ciudad("mar de plata")

Ciudad(nombre='Mar del Plata', provincia_codigo='06', provincia_nombre='Buenos Aires', poblacion_2022=682605, lat=-38.0023, lon=-57.5575)

In [11]:
arg.matching.match_ciudad("rosrio")

Ciudad(nombre='Rosario', provincia_codigo='82', provincia_nombre='Santa Fe', poblacion_2022=1028658, lat=-32.9442, lon=-60.6505)

## 5. Universidades

El score se calcula contra **sigla y nombre completo**, tomando el mejor.

In [12]:
arg.matching.match_universidad("uba")

Universidad(sigla='UBA', nombre='Universidad de Buenos Aires', provincia_codigo='02', provincia_nombre='Ciudad Autónoma de Buenos Aires', sede='Ciudad Autónoma de Buenos Aires', anio_fundacion=1821, tipo='nacional')

In [13]:
arg.matching.match_universidad("universidad d buenos aires")

Universidad(sigla='UBA', nombre='Universidad de Buenos Aires', provincia_codigo='02', provincia_nombre='Ciudad Autónoma de Buenos Aires', sede='Ciudad Autónoma de Buenos Aires', anio_fundacion=1821, tipo='nacional')

## 6. Aglomerados EPH

In [14]:
arg.matching.match_aglomerado("Gran Cordova")

Aglomerado(codigo=13, nombre='Gran Córdoba', provincia_codigo='14', provincia_nombre='Córdoba')

## 7. Función genérica

`match` contra cualquier lista de strings — útil para columnas con valores arbitrarios.

In [15]:
arg.matching.match("cordova", ["Buenos Aires", "Córdoba", "Santa Fe"])

('Córdoba', 0.8571428571428571)

In [16]:
arg.matching.candidatos(
    "cordova",
    ["Buenos Aires", "Córdoba", "Santa Fe", "Mendoza"],
    n=3,
)

[('Córdoba', 0.8571428571428571),
 ('Mendoza', 0.42857142857142855),
 ('Buenos Aires', 0.21052631578947367)]

## 8. Ajustar el umbral

Default `0.7`. Subirlo para ser estrictos, bajarlo para tolerar más ruido.

In [17]:
# Con umbral muy estricto, un typo flojo no llega.
print(arg.matching.match_provincia("misisones", umbral=0.99))

# Pero el match exacto siempre pasa (usa lookup, sin fuzzy).
print(arg.matching.match_provincia("Misiones", umbral=0.99))

None
Provincia(nombre='Misiones', codigo_indec='54', iso_id='AR-N', region='NEA', capital='Posadas', capital_lat=-27.3621, capital_lon=-55.9008, poblacion_2022=1280960, superficie_km2=29801)


## 9. Patrón de uso en una columna

Limpiar una columna con nombres sucios.

In [18]:
sucias = ["Buenos Aires", "buennos aires", "cordova", "sgo del estero", "xyz"]

for v in sucias:
    p = arg.matching.match_provincia(v)
    print(f"{v!r:25s} → {p.nombre if p else None}")

'Buenos Aires'            → Buenos Aires
'buennos aires'           → Buenos Aires
'cordova'                 → Córdoba
'sgo del estero'          → Santiago del Estero
'xyz'                     → None
